# 每个科目之下的疾病详细介绍

In [3]:
import requests
from fake_useragent import UserAgent
from bs4 import BeautifulSoup
import random
import time
from requests import exceptions
from tqdm import trange


# 创建一个UserAgent对象，用于随机生成User-Agent头
ua = UserAgent()

# 定义爬取目标URL和请求头
url = 'http://y.wksc.com/jibing/'
headers = {
    'User-Agent': ua.random,
    'Connection': 'keep-alive'
}

# 发送请求，获取响应
response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.content, 'html.parser')


In [24]:

def get_ill_intro_txt(url):
    # 定义请求头
    ua = UserAgent()
    headers = {
        'User-Agent': ua.random,
        'Connection': 'keep-alive',
    }

    # 随机延迟时间
    time.sleep(random.uniform(0.5, 1.6))

    # 发送请求，获取响应
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except (exceptions.ConnectionError, exceptions.Timeout, exceptions.RequestException):
        return None

    # 解析HTML文本，获取介绍内容
    try:
        soup = BeautifulSoup(response.content, 'html.parser')
        intro = soup.select_one('.ill-introduce-txt')
        if intro is None:
            return None
        return intro.text.strip()
    except AttributeError:
        return None

# 科目
kemu_list = soup.select('#showitem > li > a')
kemu_info_js = {
}
prefix_url = 'http://y.wksc.com'
for i, kemu in enumerate(kemu_list):
    
    kemu_info_js[kemu.text] = {
        'name' : kemu.text,
        'url' :  prefix_url + kemu['href']
        }
    kemu_select = f'.con-bd:nth-child({2*i + 2}) a'
    kemu_info_js[kemu.text]['selector'] = kemu_select
    jibing_list = soup.select(kemu_select)
    
    kemu_info_js[kemu.text]['jibing_name'] = []
    kemu_info_js[kemu.text]['jibing_url'] = []
    kemu_info_js[kemu.text]['jibing_intro'] = []

    for k in trange(len(jibing_list)):
        jibing = jibing_list[k]
        jibing_name = jibing.text
        jibing_url = prefix_url + jibing['href']
        jibing_intro = get_ill_intro_txt(jibing_url)
        
        kemu_info_js[kemu.text]['jibing_name'].append(jibing_name)
        kemu_info_js[kemu.text]['jibing_url'].append(jibing_url)
        kemu_info_js[kemu.text]['jibing_intro'].append(jibing_intro)


100%|██████████| 27/27 [01:03<00:00,  2.36s/it]


In [25]:


import os
import json
def save_js(js, save_folder = './data/', file_name = 'jibinginfo.json'):

    if not os.path.exists(save_folder):
        os.makedirs(save_folder)

    # 处理文件名中的非法字符
    for char in ['\\', '/', ':', '*', '?', '"', '<', '>', '|']:
        file_name = file_name.replace(char, '')

    
    save_path = os.path.join(save_folder, file_name)

    # 保存 JSON 文件
    try:
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(js, f, ensure_ascii=False, indent=4)
        print(f'JSON 文件已成功保存到 {save_path}。')
    except Exception as e:
        print(f'保存 JSON 文件时出错：{str(e)}')


save_js(kemu_info_js)

JSON 文件已成功保存到 ./data/jibinginfo.json。
